# 2. Basic AI Prototype with PyTorch\n
\n
## Application\n
Handwritten digit classification (MNIST) as a baseline prototype for document digitization workflows.\n
\n
## Goals\n
- Build a small neural network in PyTorch\n
- Train on MNIST\n
- Evaluate with accuracy and confusion matrix\n

In [ ]:
import torch\n
import torch.nn as nn\n
import torch.optim as optim\n
from torchvision import datasets, transforms\n
from torch.utils.data import DataLoader\n
\n
device = 'cuda' if torch.cuda.is_available() else 'cpu'\n
transform = transforms.Compose([transforms.ToTensor()])\n
train_ds = datasets.MNIST(root='./data', train=True, download=True, transform=transform)\n
test_ds = datasets.MNIST(root='./data', train=False, download=True, transform=transform)\n
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)\n
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)\n
device\n

In [ ]:
class MLP(nn.Module):\n
    def __init__(self):\n
        super().__init__()\n
        self.net = nn.Sequential(\n
            nn.Flatten(),\n
            nn.Linear(28*28, 256),\n
            nn.ReLU(),\n
            nn.Linear(256, 128),\n
            nn.ReLU(),\n
            nn.Linear(128, 10)\n
        )\n
\n
    def forward(self, x):\n
        return self.net(x)\n
\n
model = MLP().to(device)\n
criterion = nn.CrossEntropyLoss()\n
optimizer = optim.Adam(model.parameters(), lr=1e-3)\n
\n
for epoch in range(3):\n
    model.train()\n
    running_loss = 0.0\n
    for xb, yb in train_loader:\n
        xb, yb = xb.to(device), yb.to(device)\n
        optimizer.zero_grad()\n
        logits = model(xb)\n
        loss = criterion(logits, yb)\n
        loss.backward()\n
        optimizer.step()\n
        running_loss += loss.item()\n
    print(f'Epoch {epoch+1}: loss={running_loss/len(train_loader):.4f}')\n

In [ ]:
from sklearn.metrics import accuracy_score, classification_report\n
\n
model.eval()\n
preds, trues = [], []\n
with torch.no_grad():\n
    for xb, yb in test_loader:\n
        xb = xb.to(device)\n
        out = model(xb).argmax(dim=1).cpu().numpy()\n
        preds.extend(out)\n
        trues.extend(yb.numpy())\n
\n
print('Accuracy:', accuracy_score(trues, preds))\n
print(classification_report(trues, preds))\n